# Shareholder Yield Factor Index — Backtest from Pre-Loaded Data

This notebook constructs a factor-tilted index over 100 pre-selected stocks using two signals:

1. **Shareholder Yield (level)** — total payout / market cap
2. **Shareholder Yield Growth (trend)** — YoY change in total dollar payout

## Input Data Format

The notebook expects a single MultiIndex DataFrame `df` with the following structure:

- **Index level 0**: trading dates from 2016-01-01 to 2025-12-31
- **Index level 1**: stock tickers (100 constituents)
- **Columns** (6 total):
  1. `close` — closing price
  2. `market_cap` — float-adjusted market capitalization
  3. `dividends_ttm` — total dividends paid in cash over the trailing 4 quarters
  4. `buybacks_ttm` — net common stock buyback in cash over the trailing 4 quarters
  5. `total_payout_ttm` — sum of dividends and buybacks (the SY numerator)
  6. `sector` — sector classification

> **Important**: rename your columns to match these names, or update the column references in Section 1 below.

## Methodology

**Shareholder Yield:**

$$SY_{i,t} = \frac{\text{TotalPayout}_{i,t}}{\text{MarketCap}_{i,t}}$$

where $\text{TotalPayout}_{i,t}$ is the trailing-4-quarter dividends plus net buybacks for stock $i$ on date $t$.

**Shareholder Yield Growth** (measured on the dollar payout, not the ratio):

$$g_{i,t} = \frac{\text{TotalPayout}_{i,t} - \text{TotalPayout}_{i,t-252}}{\text{TotalPayout}_{i,t-252}}$$

We use 252 trading days as the YoY lookback, which approximates a calendar year. Measuring growth on the dollar numerator (rather than the yield ratio) isolates management's capital allocation decision from price noise — a 30% price drop would inflate yield-ratio growth even if the company didn't change its payout.

**Composite Score** (after cross-sectional winsorization and z-scoring at each rebalance):

$$C_{i,t} = w_{SY} \cdot z_{i,t}^{SY} + w_{gSY} \cdot z_{i,t}^{gSY}$$

**Tilt-based Weighting** against market-cap weights:

$$w_{i,t}^{index} = w_{i,t}^{mktcap} \cdot e^{\lambda C_{i,t}}$$

(then renormalize, apply position caps, and apply sector caps)

## 0. Imports and Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from datetime import datetime

import numpy as np
import pandas as pd
from scipy.stats.mstats import winsorize
import matplotlib.pyplot as plt

%matplotlib inline

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Load Your DataFrame

Replace the placeholder below with your actual data load. The expected shape is `(N_dates × N_stocks, 6 columns)` with a `(date, ticker)` MultiIndex.

After loading, the notebook standardizes column names so the rest of the code is portable. Update the `COLUMN_RENAME` dict below if your column names differ.

In [ ]:
# ─── Load your data here ───
# Replace this line with however you load your DataFrame, e.g.:
# df = pd.read_parquet("my_data.parquet")
# df = pd.read_pickle("my_data.pkl")

df = pd.read_parquet("data.parquet")  # ← replace with your actual path

# ─── Standardize column names ───
COLUMN_RENAME = {
    # left = your column name, right = expected name
    # Example: "Close": "close", "MktCap_FF": "market_cap"
}
if COLUMN_RENAME:
    df = df.rename(columns=COLUMN_RENAME)

# Expected columns after rename
EXPECTED_COLS = ["close", "market_cap", "dividends_ttm", "buybacks_ttm", "total_payout_ttm", "sector"]

# Sanity checks
assert isinstance(df.index, pd.MultiIndex), "Expected a MultiIndex (date, ticker)"
assert df.index.nlevels == 2, f"Expected 2 index levels, got {df.index.nlevels}"
df.index.names = ["date", "ticker"]

missing = set(EXPECTED_COLS) - set(df.columns)
assert not missing, f"Missing expected columns: {missing}"

# Ensure date level is a proper DatetimeIndex
if not isinstance(df.index.get_level_values("date"), pd.DatetimeIndex):
    df = df.reset_index()
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index(["date", "ticker"]).sort_index()

# Keep only expected columns in standard order
df = df[EXPECTED_COLS]

print(f"Date range:    {df.index.get_level_values('date').min().date()} to {df.index.get_level_values('date').max().date()}")
print(f"Trading days:  {df.index.get_level_values('date').nunique()}")
print(f"Tickers:       {df.index.get_level_values('ticker').nunique()}")
print(f"Total rows:    {len(df):,}")
print()
print("Sample rows:")
df.head()

In [ ]:
# Quick data quality check
print("Missing values per column:")
print(df.isna().sum())
print()
print("Sectors represented:")
print(df["sector"].dropna().unique())

## 2. Configuration

All methodology knobs live in the `Config` class. Tweak one parameter and rerun the pipeline — no hunting through the code.

In [ ]:
class Config:
    """Central configuration for the index methodology."""

    # ─── Date range ───
    BACKTEST_START = "2017-01-01"   # Allow a year of history before first rebalance
    BACKTEST_END = "2025-12-31"

    # ─── Signal parameters ───
    YOY_LOOKBACK_DAYS = 252         # Trading days for YoY growth calculation

    # ─── Cross-sectional scoring ───
    WINSORIZE_LIMITS = (0.025, 0.025)
    SIGNAL_WEIGHTS = {
        "sy_level": 0.50,
        "sy_growth": 0.50,
    }

    # ─── Portfolio construction ───
    WEIGHTING_METHOD = "tilt"       # "tilt" | "score" | "top_n"
    TOP_N = 50                      # Used only if WEIGHTING_METHOD == "top_n"
    TILT_LAMBDA = 2.0               # Exponential tilt strength
    MIN_WEIGHT = 0.0010             # Floor weight (10 bps) — higher than R1K because only 100 names
    MAX_WEIGHT = 0.08                # Cap weight at 8% (looser cap because of small universe)
    SECTOR_CAP_MULTIPLE = 2.0       # Max sector weight = benchmark × this

    # ─── Rebalancing ───
    REBALANCE_FREQ = "Q"            # "Q" = quarterly, "M" = monthly, "A" = annual

    # ─── Transaction cost ───
    COST_PER_SIDE_BPS = 15          # One-way cost in basis points

## 3. Reshape Data into Wide Panels

It's cleaner to work with one DataFrame per variable, where rows are dates and columns are tickers. We pivot the long-format DataFrame into wide panels — this lets us use vectorized operations (e.g. `payout / market_cap`) without groupby gymnastics.

In [ ]:
def pivot_panel(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """Pivot a column from the long-format MultiIndex DataFrame to a wide panel."""
    return df[column].unstack(level="ticker").sort_index()


close = pivot_panel(df, "close")
market_cap = pivot_panel(df, "market_cap")
dividends = pivot_panel(df, "dividends_ttm")
buybacks = pivot_panel(df, "buybacks_ttm")
total_payout = pivot_panel(df, "total_payout_ttm")

# Sector is a string and (typically) doesn't change much over time — take the most recent
sector_panel = pivot_panel(df, "sector")
# For mapping purposes, take the last non-null sector for each ticker
sectors = sector_panel.ffill().iloc[-1]

print(f"Panel shape (close):       {close.shape}")
print(f"Panel shape (market_cap):  {market_cap.shape}")
print(f"Panel shape (total_payout): {total_payout.shape}")
print(f"Sectors mapped for {sectors.notna().sum()}/{len(sectors)} tickers")

## 4. Build the Two Signals as Daily Panels

### Shareholder Yield (Level)

$$SY_{i,t} = \frac{\text{TotalPayout}_{i,t}}{\text{MarketCap}_{i,t}}$$

### Shareholder Yield Growth

$$g_{i,t} = \frac{\text{TotalPayout}_{i,t} - \text{TotalPayout}_{i,t-252}}{\text{TotalPayout}_{i,t-252}}$$

Negative or zero prior-period payouts make the growth ratio undefined or misleading, so we mask those out. Stocks with no prior-period payout effectively get treated as missing on the growth signal — they're only scored on the level signal.

In [ ]:
# ─── Shareholder yield (level) ───
sy_level = total_payout / market_cap.replace(0, np.nan)
sy_level = sy_level.replace([np.inf, -np.inf], np.nan)

# ─── Shareholder yield growth (YoY on the dollar payout) ───
prior_payout = total_payout.shift(Config.YOY_LOOKBACK_DAYS)
# Mask non-positive prior payouts so growth is well-defined
prior_payout_clean = prior_payout.where(prior_payout > 0)
sy_growth = (total_payout - prior_payout_clean) / prior_payout_clean
sy_growth = sy_growth.replace([np.inf, -np.inf], np.nan)

print(f"SY level panel:   {sy_level.shape}")
print(f"SY growth panel:  {sy_growth.shape}")
print()
print("SY level summary (last available date):")
print(sy_level.iloc[-1].describe())
print()
print("SY growth summary (last available date):")
print(sy_growth.iloc[-1].describe())

### Distribution Check

Eyeball the cross-sectional distributions on a sample date. Expect heavy right tails on both — a few high-payout names and a few names with explosive growth from a low base.

In [ ]:
sample_date = sy_level.dropna(how="all").index[-1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(sy_level.loc[sample_date].dropna().clip(-0.05, 0.20), bins=30,
             color="#2563eb", edgecolor="white")
axes[0].set_title(f"Shareholder Yield — {sample_date.date()}")
axes[0].set_xlabel("Shareholder Yield")
axes[0].set_ylabel("Count")
axes[0].axvline(sy_level.loc[sample_date].median(), color="red", linestyle="--",
                label=f"Median: {sy_level.loc[sample_date].median():.2%}")
axes[0].legend()

axes[1].hist(sy_growth.loc[sample_date].dropna().clip(-2, 5), bins=30,
             color="#059669", edgecolor="white")
axes[1].set_title(f"SY Growth (YoY $ payout) — {sample_date.date()}")
axes[1].set_xlabel("YoY Payout Growth")
axes[1].set_ylabel("Count")
axes[1].axvline(sy_growth.loc[sample_date].median(), color="red", linestyle="--",
                label=f"Median: {sy_growth.loc[sample_date].median():.2%}")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Cross-Sectional Z-Scoring

For each rebalance date, we winsorize at the 2.5th/97.5th percentiles, then z-score:

$$z_{i,t}^{k} = \frac{x_{i,t}^{k} - \bar{x}_t^{k}}{\sigma_t^{k}}$$

This makes the level and growth signals directly comparable on the same scale.

In [ ]:
def cross_sectional_zscore(signal: pd.Series, limits: tuple = Config.WINSORIZE_LIMITS) -> pd.Series:
    """Winsorize then z-score a single cross-section of stocks."""
    valid = signal.dropna()
    if len(valid) < 10:
        return signal * np.nan

    winsorized = pd.Series(
        winsorize(valid.values, limits=limits),
        index=valid.index,
        dtype=float,
    )
    mu = winsorized.mean()
    sigma = winsorized.std()
    if sigma < 1e-12:
        return winsorized * 0.0

    z = (winsorized - mu) / sigma
    return z.reindex(signal.index)


def build_composite_score(
    z_sy: pd.Series, z_sy_growth: pd.Series, weights: dict = Config.SIGNAL_WEIGHTS
) -> pd.Series:
    """Combine z-scored signals into a single composite, weighting by SIGNAL_WEIGHTS."""
    composite = (
        weights["sy_level"] * z_sy.fillna(0)
        + weights["sy_growth"] * z_sy_growth.fillna(0)
    )
    # Penalize stocks missing both signals
    both_missing = z_sy.isna() & z_sy_growth.isna()
    composite[both_missing] = np.nan
    return composite

## 6. Portfolio Construction

Three weighting methods (set via `Config.WEIGHTING_METHOD`):

**`"tilt"`** — Start from market-cap weights, tilt exponentially:

$$w_i = \frac{w_i^{mktcap} \cdot e^{\lambda C_i}}{\sum_j w_j^{mktcap} \cdot e^{\lambda C_j}}$$

**`"score"`** — Weight purely by composite score (floored at zero), more aggressive.

**`"top_n"`** — Pick top N stocks, market-cap weight within that subset.

After initial weighting we apply single-name caps (`MIN_WEIGHT`, `MAX_WEIGHT`) and sector caps.

In [ ]:
def compute_index_weights(
    composite: pd.Series,
    mktcap: pd.Series,
    sectors: pd.Series,
    method: str = Config.WEIGHTING_METHOD,
) -> pd.Series:
    """Convert composite scores into normalized portfolio weights."""
    valid = composite.dropna().index.intersection(mktcap.dropna().index)
    comp = composite.loc[valid]
    mc = mktcap.loc[valid]
    sec = sectors.reindex(valid)

    mc_weights = mc / mc.sum()

    if method == "tilt":
        tilt_factor = np.exp(Config.TILT_LAMBDA * comp)
        raw_weights = mc_weights * tilt_factor
    elif method == "score":
        raw_weights = comp.clip(lower=0)
    elif method == "top_n":
        top_tickers = comp.nlargest(Config.TOP_N).index
        raw_weights = mc_weights.reindex(top_tickers).fillna(0)
    else:
        raise ValueError(f"Unknown weighting method: {method}")

    if raw_weights.sum() == 0:
        return pd.Series(dtype=float)
    weights = raw_weights / raw_weights.sum()

    # Single-name caps
    weights = weights.clip(upper=Config.MAX_WEIGHT)
    weights = weights[weights >= Config.MIN_WEIGHT]
    if weights.sum() == 0:
        return pd.Series(dtype=float)
    weights = weights / weights.sum()

    # Sector caps
    if Config.SECTOR_CAP_MULTIPLE is not None and sec.notna().any():
        benchmark_sector_wt = mc.groupby(sec).sum() / mc.sum()
        sector_caps = benchmark_sector_wt * Config.SECTOR_CAP_MULTIPLE

        for _ in range(5):  # Iterate a few times — clipping one sector may push others over
            adjusted = False
            for sector in sec.dropna().unique():
                mask = sec.reindex(weights.index) == sector
                sector_wt = weights[mask].sum()
                cap = sector_caps.get(sector, 1.0)
                if sector_wt > cap and sector_wt > 0:
                    weights[mask] *= (cap / sector_wt)
                    adjusted = True
            weights = weights / weights.sum()
            if not adjusted:
                break

    return weights

## 7. Build the Rebalance Schedule and Weight History

For each rebalance date in the backtest window, we:

1. Pull the cross-section of `sy_level` and `sy_growth` values on that date
2. Winsorize and z-score them
3. Combine into a composite
4. Convert to portfolio weights
5. Store in `weights_history[rebal_date]`

Quarterly rebalancing (`Config.REBALANCE_FREQ = "Q"`) is the default — this matches industry convention for factor indices and balances signal freshness against turnover.

In [ ]:
def build_rebalance_dates(start: str, end: str, freq: str, all_trading_days: pd.DatetimeIndex) -> pd.DatetimeIndex:
    """
    Build a list of rebalance dates by snapping period-end dates to the nearest prior trading day.
    """
    period_ends = pd.date_range(start=start, end=end, freq=freq)
    rebal_dates = []
    for pe in period_ends:
        # Snap to the nearest trading day on or before the period end
        eligible = all_trading_days[all_trading_days <= pe]
        if len(eligible) > 0:
            rebal_dates.append(eligible[-1])
    return pd.DatetimeIndex(sorted(set(rebal_dates)))


trading_days = close.index
rebal_dates = build_rebalance_dates(
    Config.BACKTEST_START,
    Config.BACKTEST_END,
    Config.REBALANCE_FREQ,
    trading_days,
)
print(f"Rebalance dates: {len(rebal_dates)}")
print(f"First:           {rebal_dates[0].date()}")
print(f"Last:            {rebal_dates[-1].date()}")
print(f"Sample (first 5): {[d.date() for d in rebal_dates[:5]]}")

In [ ]:
# Build weights for each rebalance date
weights_history = {}
composite_history = {}

for rebal_date in rebal_dates:
    sy_xsec = sy_level.loc[rebal_date]
    g_xsec = sy_growth.loc[rebal_date]
    mc_xsec = market_cap.loc[rebal_date]

    z_sy = cross_sectional_zscore(sy_xsec)
    z_g = cross_sectional_zscore(g_xsec)
    composite = build_composite_score(z_sy, z_g)

    weights = compute_index_weights(composite, mc_xsec, sectors)

    if len(weights) > 0:
        weights_history[rebal_date] = weights
        composite_history[rebal_date] = composite

print(f"Built weights for {len(weights_history)} rebalance dates")
print()
print(f"Latest rebalance ({list(weights_history.keys())[-1].date()}):")
latest_w = weights_history[list(weights_history.keys())[-1]]
print(f"  Holdings:   {len(latest_w)}")
print(f"  Max weight: {latest_w.max():.2%}")
print(f"  Min weight: {latest_w.min():.2%}")
print(f"  Sum check:  {latest_w.sum():.6f}")

### Inspect the Latest Portfolio

In [ ]:
latest_date = list(weights_history.keys())[-1]
latest_weights = weights_history[latest_date]
latest_composite = composite_history[latest_date]

print(f"Top 15 holdings as of {latest_date.date()}:")
print(f"{'Ticker':<10} {'Weight':>8} {'Composite':>11} {'SY Level':>10} {'SY Growth':>11} {'Sector':<25}")
print("-" * 80)
for ticker, wt in latest_weights.nlargest(15).items():
    sy_v = sy_level.loc[latest_date, ticker]
    g_v = sy_growth.loc[latest_date, ticker]
    cs = latest_composite.get(ticker, np.nan)
    sec = sectors.get(ticker, "Unknown")
    print(f"{ticker:<10} {wt:>7.2%} {cs:>+11.2f} {sy_v:>9.2%} {g_v:>+10.2%} {str(sec)[:24]:<25}")

## 8. Backtest Engine

For each rebalance date:

1. Compute one-way turnover vs. previous weights and deduct transaction costs
2. Compute daily portfolio returns from rebalance date until next rebalance
3. Let weights drift with prices between rebalances (buy-and-hold)
4. Stitch all the periods together into a continuous return series

### Transaction Cost Model

$$TC_{rebal} = \tau \cdot c \quad \text{where} \quad \tau = \frac{1}{2}\sum_i |w_{i,new} - w_{i,old}|$$

The sum of absolute weight changes equals 2× one-way turnover, so we divide by 2 and multiply by cost per side.

### Benchmark

Since you have 100 stocks selected, the most apples-to-apples benchmark is a **market-cap-weighted index of those same 100 stocks**. This isolates the factor-tilting effect from the universe-selection effect. If you wanted to instead compare against the broader Russell 1000, that comparison includes both effects bundled together.

In [ ]:
def estimate_transaction_costs(old_weights: pd.Series, new_weights: pd.Series) -> float:
    """Compute the cost drag on a single rebalance."""
    all_tickers = old_weights.index.union(new_weights.index)
    old = old_weights.reindex(all_tickers, fill_value=0)
    new = new_weights.reindex(all_tickers, fill_value=0)
    one_way_turnover = (new - old).abs().sum() / 2
    return one_way_turnover * (Config.COST_PER_SIDE_BPS / 10_000)


def run_backtest(
    weights_history: dict,
    price_panel: pd.DataFrame,
    benchmark_returns: pd.Series,
) -> pd.DataFrame:
    """
    Simulate factor index returns. Returns a DataFrame with:
      - factor_index: cumulative return of the factor portfolio (starts at 1)
      - benchmark:    cumulative return of the benchmark (starts at 1)
      - excess:       factor_index / benchmark
    """
    rebal_dates = sorted(weights_history.keys())
    all_dates = price_panel.index.sort_values()

    daily_ret_records = []
    prev_weights = pd.Series(dtype=float)

    for i, rebal_date in enumerate(rebal_dates):
        target_weights = weights_history[rebal_date]
        tc = estimate_transaction_costs(prev_weights, target_weights)

        # Determine date range for this holding period
        if rebal_date not in all_dates:
            continue
        start_idx = all_dates.get_loc(rebal_date)
        if i + 1 < len(rebal_dates):
            next_rebal = rebal_dates[i + 1]
            if next_rebal in all_dates:
                end_idx = all_dates.get_loc(next_rebal)
            else:
                end_idx = len(all_dates)
        else:
            end_idx = len(all_dates)

        period_dates = all_dates[start_idx:end_idx + 1]
        if len(period_dates) < 2:
            continue

        held_tickers = target_weights.index.intersection(price_panel.columns)
        period_prices = price_panel.loc[period_dates, held_tickers]
        daily_returns = period_prices.pct_change()

        # Initialize weights at start of period
        w = target_weights.reindex(held_tickers, fill_value=0)
        w = w / w.sum() if w.sum() > 0 else w

        for j, date in enumerate(period_dates[1:], 1):
            day_ret = daily_returns.loc[date].fillna(0)
            port_ret = (w * day_ret.reindex(w.index, fill_value=0)).sum()

            if j == 1:
                port_ret -= tc  # Deduct TC on rebalance day

            daily_ret_records.append({"date": date, "factor_return": port_ret})

            # Drift weights between rebalances
            w = w * (1 + day_ret.reindex(w.index, fill_value=0))
            wsum = w.sum()
            if wsum > 0:
                w = w / wsum

        prev_weights = target_weights

    ret_df = pd.DataFrame(daily_ret_records).set_index("date")
    ret_df.index = pd.DatetimeIndex(ret_df.index)
    ret_df = ret_df[~ret_df.index.duplicated(keep="last")]

    bmk = benchmark_returns.reindex(ret_df.index).fillna(0)

    results = pd.DataFrame({
        "factor_index": (1 + ret_df["factor_return"]).cumprod(),
        "benchmark": (1 + bmk).cumprod(),
    })
    results["excess"] = results["factor_index"] / results["benchmark"]
    return results

### Build the Custom 100-Stock Benchmark

The benchmark holds the same 100 stocks at market-cap weights, rebalanced quarterly to match the factor index's rebalance cadence.

In [ ]:
def build_mktcap_benchmark_returns(
    market_cap: pd.DataFrame,
    price_panel: pd.DataFrame,
    rebal_dates: pd.DatetimeIndex,
) -> pd.Series:
    """
    Build daily returns for a market-cap-weighted index of the same universe,
    rebalanced on the same schedule as the factor index.
    """
    all_dates = price_panel.index.sort_values()
    daily_records = []

    for i, rebal_date in enumerate(rebal_dates):
        if rebal_date not in all_dates:
            continue
        mc_xsec = market_cap.loc[rebal_date].dropna()
        if mc_xsec.sum() == 0:
            continue
        bench_w = mc_xsec / mc_xsec.sum()

        start_idx = all_dates.get_loc(rebal_date)
        if i + 1 < len(rebal_dates):
            next_rebal = rebal_dates[i + 1]
            if next_rebal in all_dates:
                end_idx = all_dates.get_loc(next_rebal)
            else:
                end_idx = len(all_dates)
        else:
            end_idx = len(all_dates)

        period_dates = all_dates[start_idx:end_idx + 1]
        if len(period_dates) < 2:
            continue

        held = bench_w.index.intersection(price_panel.columns)
        period_prices = price_panel.loc[period_dates, held]
        daily_returns = period_prices.pct_change()

        w = bench_w.reindex(held, fill_value=0)
        w = w / w.sum() if w.sum() > 0 else w

        for date in period_dates[1:]:
            day_ret = daily_returns.loc[date].fillna(0)
            port_ret = (w * day_ret.reindex(w.index, fill_value=0)).sum()
            daily_records.append({"date": date, "benchmark_return": port_ret})

            w = w * (1 + day_ret.reindex(w.index, fill_value=0))
            wsum = w.sum()
            if wsum > 0:
                w = w / wsum

    bmk_df = pd.DataFrame(daily_records).set_index("date")
    bmk_df.index = pd.DatetimeIndex(bmk_df.index)
    bmk_df = bmk_df[~bmk_df.index.duplicated(keep="last")]
    return bmk_df["benchmark_return"]


benchmark_returns = build_mktcap_benchmark_returns(market_cap, close, rebal_dates)
print(f"Benchmark return series: {len(benchmark_returns)} daily observations")
print(f"From {benchmark_returns.index.min().date()} to {benchmark_returns.index.max().date()}")

In [ ]:
# Run the backtest
results = run_backtest(weights_history, close, benchmark_returns)
print(f"Backtest produced {len(results)} daily observations")
print()
print("Last 5 days:")
results.tail()

## 9. Performance Analytics

### Metrics Computed

- **Annualized Return** — $(1 + \bar{r}_{daily})^{252} - 1$
- **Annualized Volatility** — $\sigma_{daily} \cdot \sqrt{252}$
- **Sharpe Ratio** — annualized return / annualized volatility (assuming zero risk-free rate)
- **Max Drawdown** — largest peak-to-trough decline
- **Total Return** — cumulative return over the full backtest window
- **Information Ratio** — annualized excess return / annualized tracking error

In [ ]:
def compute_performance_stats(results: pd.DataFrame) -> pd.DataFrame:
    """Compute headline performance metrics for factor index, benchmark, and excess."""
    factor_ret = results["factor_index"].pct_change().dropna()
    bmk_ret = results["benchmark"].pct_change().dropna()
    excess_ret = factor_ret - bmk_ret

    trading_days = 252

    def stats_for(rets):
        ann_ret = (1 + rets.mean()) ** trading_days - 1
        ann_vol = rets.std() * np.sqrt(trading_days)
        sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
        cum = (1 + rets).cumprod()
        max_dd = (cum / cum.cummax() - 1).min()
        total_ret = cum.iloc[-1] - 1
        return {
            "Annualized Return": ann_ret,
            "Annualized Volatility": ann_vol,
            "Sharpe Ratio": sharpe,
            "Max Drawdown": max_dd,
            "Total Return": total_ret,
        }

    df_stats = pd.DataFrame({
        "Factor Index": stats_for(factor_ret),
        "Benchmark":    stats_for(bmk_ret),
        "Excess":       stats_for(excess_ret),
    })

    # Information ratio = ann excess return / ann tracking error
    tracking_error = excess_ret.std() * np.sqrt(trading_days)
    info_ratio = (excess_ret.mean() * trading_days) / tracking_error if tracking_error > 0 else 0
    df_stats.loc["Information Ratio"] = ["", "", info_ratio]
    df_stats.loc["Tracking Error"] = ["", "", tracking_error]

    return df_stats


stats_df = compute_performance_stats(results)
# Format display
def _fmt(x):
    if isinstance(x, str):
        return x
    if abs(x) < 10 and not isinstance(x, bool):
        return f"{x:.2%}" if abs(x) < 1 else f"{x:.2f}"
    return f"{x:.2f}"

print("Performance Summary:")
print(stats_df.applymap(_fmt))

In [ ]:
def compute_turnover_stats(weights_history: dict) -> dict:
    """Compute turnover statistics across all rebalances."""
    dates = sorted(weights_history.keys())
    turnovers = []

    for i in range(1, len(dates)):
        old_w = weights_history[dates[i - 1]]
        new_w = weights_history[dates[i]]
        all_tickers = old_w.index.union(new_w.index)
        t = (
            new_w.reindex(all_tickers, fill_value=0)
            - old_w.reindex(all_tickers, fill_value=0)
        ).abs().sum() / 2
        turnovers.append(t)

    rebal_per_year = {"Q": 4, "M": 12, "A": 1}.get(Config.REBALANCE_FREQ, 4)
    avg_turnover = np.mean(turnovers) if turnovers else 0
    annual_tc_drag = avg_turnover * rebal_per_year * Config.COST_PER_SIDE_BPS / 10_000

    return {
        "Mean One-Way Turnover": f"{avg_turnover:.2%}",
        "Max One-Way Turnover":  f"{np.max(turnovers):.2%}" if turnovers else "N/A",
        "Min One-Way Turnover":  f"{np.min(turnovers):.2%}" if turnovers else "N/A",
        "Rebalance Count":        len(dates),
        "Est. Annual TC Drag":    f"{annual_tc_drag:.4%}",
    }


turnover_stats = compute_turnover_stats(weights_history)
print("Turnover Statistics:")
for k, v in turnover_stats.items():
    print(f"  {k:<28} {v}")

## 10. Visualizations

### Cumulative Return + Relative Performance

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), height_ratios=[2, 1])
fig.suptitle("Shareholder Yield Factor Index vs Mkt-Cap-Weighted Benchmark (Same 100 Stocks)",
             fontsize=13, fontweight="bold")

ax1.plot(results.index, results["factor_index"], label="SY Factor Index",
         linewidth=1.6, color="#2563eb")
ax1.plot(results.index, results["benchmark"], label="Mkt-Cap Benchmark",
         linewidth=1.6, color="#6b7280", linestyle="--")
ax1.set_ylabel("Growth of $1")
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)

ax2.plot(results.index, results["excess"], label="Factor / Benchmark",
         linewidth=1.5, color="#059669")
ax2.axhline(y=1.0, color="#6b7280", linestyle=":", linewidth=0.8)
ax2.set_ylabel("Relative Performance")
ax2.set_xlabel("Date")
ax2.legend(loc="upper left")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Drawdown Comparison

A drawdown panel often surfaces information that the cumulative return chart hides — particularly whether the factor is taking on additional tail risk versus the benchmark.

In [ ]:
factor_dd = results["factor_index"] / results["factor_index"].cummax() - 1
bench_dd = results["benchmark"] / results["benchmark"].cummax() - 1

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(results.index, factor_dd * 100, 0, color="#2563eb", alpha=0.4, label="SY Factor Index")
ax.fill_between(results.index, bench_dd * 100, 0, color="#6b7280", alpha=0.4, label="Benchmark")
ax.set_ylabel("Drawdown (%)")
ax.set_xlabel("Date")
ax.set_title("Drawdown Comparison")
ax.legend(loc="lower left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Rolling 12-Month Excess Return

How consistent is the outperformance? Constant outperformance is rare; more common is a mix of strong years and weak years. Watch for whether the strategy has long stretches of underperformance — that's a real-world test of whether you'd actually stick with it.

In [ ]:
factor_ret = results["factor_index"].pct_change()
bench_ret = results["benchmark"].pct_change()
excess_ret = factor_ret - bench_ret

rolling_excess = (1 + excess_ret).rolling(252).apply(np.prod, raw=True) - 1

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(rolling_excess.index, rolling_excess * 100, 0,
                where=(rolling_excess > 0), color="#059669", alpha=0.5, label="Outperformance")
ax.fill_between(rolling_excess.index, rolling_excess * 100, 0,
                where=(rolling_excess <= 0), color="#dc2626", alpha=0.5, label="Underperformance")
ax.axhline(y=0, color="#6b7280", linewidth=0.8)
ax.set_ylabel("Rolling 12-Month Excess Return (%)")
ax.set_xlabel("Date")
ax.set_title("Rolling 12-Month Excess Return: Factor vs Benchmark")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Turnover Over Time

Spikes in turnover often correspond to market regime shifts. If turnover is dramatically higher in volatile periods, that's where most of the transaction cost drag is hiding.

In [ ]:
rebal_list = sorted(weights_history.keys())
turnover_series = []
for i in range(1, len(rebal_list)):
    old_w = weights_history[rebal_list[i - 1]]
    new_w = weights_history[rebal_list[i]]
    all_t = old_w.index.union(new_w.index)
    t = (new_w.reindex(all_t, fill_value=0) - old_w.reindex(all_t, fill_value=0)).abs().sum() / 2
    turnover_series.append((rebal_list[i], t))

turnover_df = pd.DataFrame(turnover_series, columns=["date", "turnover"]).set_index("date")

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(turnover_df.index, turnover_df["turnover"] * 100, width=60, color="#7c3aed", alpha=0.7)
ax.axhline(y=turnover_df["turnover"].mean() * 100, color="red", linestyle="--",
           label=f"Mean: {turnover_df['turnover'].mean():.1%}")
ax.set_ylabel("One-Way Turnover (%)")
ax.set_xlabel("Rebalance Date")
ax.set_title("Quarterly One-Way Turnover")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

### Sector Allocation Snapshot

In [ ]:
latest_date = list(weights_history.keys())[-1]
latest_w = weights_history[latest_date]

# Factor index sector allocation
factor_sec = latest_w.groupby(sectors.reindex(latest_w.index)).sum().sort_values(ascending=False)

# Benchmark sector allocation on same date
mc_latest = market_cap.loc[latest_date].dropna()
bench_w = mc_latest / mc_latest.sum()
bench_sec = bench_w.groupby(sectors.reindex(bench_w.index)).sum().reindex(factor_sec.index)

sector_df = pd.DataFrame({
    "Factor Index": factor_sec,
    "Benchmark":    bench_sec,
}).fillna(0)
sector_df["Active Weight"] = sector_df["Factor Index"] - sector_df["Benchmark"]

print(f"Sector Allocation as of {latest_date.date()}:")
print((sector_df * 100).round(2))

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(sector_df))
width = 0.35
ax.bar(x - width/2, sector_df["Factor Index"] * 100, width, label="Factor Index", color="#2563eb")
ax.bar(x + width/2, sector_df["Benchmark"] * 100, width, label="Benchmark", color="#6b7280")
ax.set_xticks(x)
ax.set_xticklabels(sector_df.index, rotation=45, ha="right")
ax.set_ylabel("Weight (%)")
ax.set_title(f"Sector Allocation: Factor Index vs Benchmark ({latest_date.date()})")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 11. Next Steps

A few extensions worth considering once the baseline is working:

**Sensitivity analysis on Config parameters.** Loop over different `TILT_LAMBDA` values, `SIGNAL_WEIGHTS` combinations, or `REBALANCE_FREQ` choices and compare Sharpe ratios. This tells you which knobs are actually doing work versus which are vestigial.

**Signal IC analysis.** Compute the Spearman rank correlation between each signal at $t$ and the cross-sectional forward return over $[t, t+1Q]$. A median IC of 0.02–0.05 is solid; below 0.01 means your signal isn't really predictive and the strong backtest may be coming from somewhere else.

**Quantile spread analysis.** Sort stocks each quarter into composite-score quintiles. If Q5 (best scores) consistently outperforms Q1 (worst scores), the signal is doing what you want. If only Q5 outperforms while Q4–Q1 are noise, the signal is concentrated in the tails.

**Factor decomposition.** Regress your daily excess returns against known factors (Mkt, SMB, HML, RMW, CMA from Ken French's data library) using statsmodels. The intercept (alpha) tells you whether the strategy is adding something beyond known premiums. A high R² with significant loadings on HML and RMW would suggest you've largely rediscovered value and quality.

**Compare to a "growth-only" and "level-only" version.** Run the same backtest with `SIGNAL_WEIGHTS = {"sy_level": 1.0, "sy_growth": 0.0}` and the reverse. If the combined version doesn't beat the better of the two, the diversification benefit isn't real and you might as well simplify.